# Champion Profile: Where Spain Ranked

## Purpose
This notebook profiles the 2026 FIFA World Cup champion, Spain, using FIFA official Match Centre statistics.

The goal is not to claim that every high ranking caused Spain to win. Instead, the analysis places the champion's tactical profile in tournament context:

- Where did Spain rank in territorial control?
- Did Spain turn territory into shots?
- Did Spain progress through defensive lines?
- Which final-third zones did Spain use most?
- What did Spain's off-ball receiving profile look like?

## Data Scope
- Source file: `site_official_stats_team_wide_flagged.csv`
- Main filter: `stats_complete == True`
- Analysis sample: matches with full FIFA Official Stats only
- Excluded match: Belgium vs Egypt (`match_id = 400021478`), because FIFA only provides Live Statistics for that match
- Champion: Spain, based on FIFA's final tournament standings and final report

## Interpretation Rule
Rankings are descriptive. They show where Spain stood relative to other teams in the dataset. They do not prove causality.

## Data Limitations for Publication
- Team match counts differ across the tournament because some teams played 3 matches and finalists played up to 8 matches. Spain-centered rankings are descriptive champion profiling, not a causal model of why Spain won.
- Belgium vs Egypt (`match_id = 400021478`) is excluded from the main analytical tables because FIFA provides only `Live Statistics` for that match. As a result, Belgium has 5 full-stat matches instead of 6, and Egypt has 4 full-stat matches instead of 5. Per-match metrics for those two teams can be slightly inflated because one real match is not in the denominator.




In [ ]:
"""
Step 1: Environment setup and data load
=======================================
Only BASE_DIR should need editing if the project folder moves.
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

BASE_DIR = Path.cwd().resolve()
for parent in [BASE_DIR, *BASE_DIR.parents]:
    if parent.name == "worldcup-2026-official-stats-analysis":
        BASE_DIR = parent
        break
DATA_DIR = BASE_DIR / "data" / "fifa_worldcup_2026" / "site_scrape"
PUBLIC_DIR = BASE_DIR
OUTPUT_DATA_DIR = PUBLIC_DIR / "data"
OUTPUT_FIG_DIR = PUBLIC_DIR / "figures"

RAW_CSV_PATH = DATA_DIR / "site_official_stats_team_wide_flagged.csv"
GROUP_STAGE_PATH = DATA_DIR / "site_official_stats_team_wide_group_stage.csv"

for path_name, path_value in {
    "BASE_DIR": BASE_DIR,
    "DATA_DIR": DATA_DIR,
    "PUBLIC_DIR": PUBLIC_DIR,
    "RAW_CSV_PATH": RAW_CSV_PATH,
    "GROUP_STAGE_PATH": GROUP_STAGE_PATH,
}.items():
    if not path_value.exists():
        raise FileNotFoundError(f"{path_name} does not exist: {path_value}")

OUTPUT_DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_FIG_DIR.mkdir(parents=True, exist_ok=True)

raw = pd.read_csv(RAW_CSV_PATH)

print("[Raw file]")
print(f"Rows: {len(raw):,}")
print(f"Matches: {raw['match_id'].nunique():,}")
print(f"Teams: {raw['team_name'].nunique():,}")

display(
    raw[["match_id", "stats_source", "stats_complete"]]
    .drop_duplicates()
    .groupby(["stats_source", "stats_complete"])
    .size()
    .reset_index(name="matches")
)

# Keep only full FIFA Official Stats matches. Missing values are not imputed.
df = raw[raw["stats_complete"] == True].copy()

print("\n[Analysis file]")
print(f"Rows: {len(df):,}")
print(f"Matches: {df['match_id'].nunique():,}")
print(f"Teams: {df['team_name'].nunique():,}")




## Metric Design

The champion profile uses two types of indicators.

**Performance and efficiency indicators** are interpreted as stronger when the value is higher:

- Field Tilt Proxy
- Shot Creation Efficiency
- Goals per Attempt
- Line Break Completion Rate
- Defensive Line Break Completion Rate
- Completed Defensive Line Breaks per Match
- Final Third Entries per Match
- Attempts at Goal per Match
- Pressing-to-Turnover Rate

**Style indicators** describe tactical tendency rather than quality by themselves:

- Wide Entry Share
- Half-space Entry Share
- Central Entry Share
- In-behind Offer Share
- Between-lines Offer Share
- In-front Offer Share

A high style ranking means Spain used that route or movement pattern more often than most teams. It does not automatically mean better performance.


In [ ]:
"""
Step 2: Create phase labels and derive team-level metrics
=========================================================
Team metrics are calculated from totals within each phase, not from averages of match-level ratios.
"""

CHAMPION_TEAM = "Spain"

group_stage_match_ids = set(pd.read_csv(GROUP_STAGE_PATH)["match_id"].unique())

df["competition_phase"] = np.where(
    df["match_id"].isin(group_stage_match_ids),
    "Group Stage",
    "Knockout Stage"
)

print("[Phase split]")
display(
    df[["match_id", "competition_phase"]]
    .drop_duplicates()
    .groupby("competition_phase")
    .size()
    .reset_index(name="matches")
)

final_third_cols = [
    "attacking__final_third_entries__left_channel",
    "attacking__final_third_entries__left_inside_channel",
    "attacking__final_third_entries__central_channel",
    "attacking__final_third_entries__right_inside_channel",
    "attacking__final_third_entries__right_channel",
]

required_cols = final_third_cols + [
    "match_id",
    "team_name",
    "team_side",
    "opponent_side",
    "attacking__goal__total",
    "attacking__attempts_at_goal__total",
    "attacking__offers_to_receive__total",
    "attacking__offers_to_receive__in_behind",
    "attacking__offers_to_receive__in_between",
    "attacking__offers_to_receive__in_front",
    "attacking__offers_to_receive__receptions_between_midfield_and_defensive_lines",
    "attacking__offers_to_receive__receptions_behind_the_defensive_line",
    "attacking__line_breaks__attempted_line_breaks",
    "attacking__line_breaks__completed_line_breaks",
    "attacking__line_breaks__attempted_defensive_line_breaks",
    "attacking__line_breaks__completed_defensive_line_breaks",
    "defending__forced_turnovers",
    "defending__pressing_applied",
]

missing_required = [col for col in required_cols if col not in df.columns]
if missing_required:
    raise ValueError(f"Missing required columns: {missing_required}")


def safe_divide(numerator, denominator):
    return np.where(denominator == 0, np.nan, numerator / denominator)


def calculate_team_metrics(input_df, phase_label):
    temp = input_df.copy()

    temp["final_third_entries_total"] = temp[final_third_cols].sum(axis=1, min_count=5)
    temp["wide_entries"] = (
        temp["attacking__final_third_entries__left_channel"]
        + temp["attacking__final_third_entries__right_channel"]
    )
    temp["halfspace_entries"] = (
        temp["attacking__final_third_entries__left_inside_channel"]
        + temp["attacking__final_third_entries__right_inside_channel"]
    )
    temp["central_entries"] = temp["attacking__final_third_entries__central_channel"]

    opponent_f3 = (
        temp[["match_id", "team_side", "final_third_entries_total"]]
        .rename(columns={
            "team_side": "opponent_side",
            "final_third_entries_total": "opponent_final_third_entries_total",
        })
    )

    temp = temp.merge(
        opponent_f3,
        on=["match_id", "opponent_side"],
        how="left",
        validate="many_to_one",
    )

    valid = temp.dropna(subset=["final_third_entries_total", "opponent_final_third_entries_total"]).copy()

    team = (
        valid
        .groupby("team_name")
        .agg(
            matches=("match_id", "nunique"),
            final_third_entries_total=("final_third_entries_total", "sum"),
            opponent_final_third_entries_total=("opponent_final_third_entries_total", "sum"),
            wide_entries=("wide_entries", "sum"),
            halfspace_entries=("halfspace_entries", "sum"),
            central_entries=("central_entries", "sum"),
            attempts_at_goal=("attacking__attempts_at_goal__total", "sum"),
            goals=("attacking__goal__total", "sum"),
            attempted_line_breaks=("attacking__line_breaks__attempted_line_breaks", "sum"),
            completed_line_breaks=("attacking__line_breaks__completed_line_breaks", "sum"),
            attempted_defensive_line_breaks=("attacking__line_breaks__attempted_defensive_line_breaks", "sum"),
            completed_defensive_line_breaks=("attacking__line_breaks__completed_defensive_line_breaks", "sum"),
            offers_total=("attacking__offers_to_receive__total", "sum"),
            offers_in_behind=("attacking__offers_to_receive__in_behind", "sum"),
            offers_in_between=("attacking__offers_to_receive__in_between", "sum"),
            offers_in_front=("attacking__offers_to_receive__in_front", "sum"),
            receptions_between_lines=("attacking__offers_to_receive__receptions_between_midfield_and_defensive_lines", "sum"),
            receptions_behind_line=("attacking__offers_to_receive__receptions_behind_the_defensive_line", "sum"),
            pressing_applied=("defending__pressing_applied", "sum"),
            forced_turnovers=("defending__forced_turnovers", "sum"),
        )
        .reset_index()
    )

    team["competition_phase"] = phase_label

    team["field_tilt_proxy"] = safe_divide(
        team["final_third_entries_total"],
        team["final_third_entries_total"] + team["opponent_final_third_entries_total"]
    ) * 100
    team["shot_creation_efficiency"] = safe_divide(team["attempts_at_goal"], team["final_third_entries_total"]) * 100
    team["goals_per_attempt"] = safe_divide(team["goals"], team["attempts_at_goal"]) * 100

    team["final_third_entries_per_match"] = safe_divide(team["final_third_entries_total"], team["matches"])
    team["attempts_at_goal_per_match"] = safe_divide(team["attempts_at_goal"], team["matches"])
    team["completed_line_breaks_per_match"] = safe_divide(team["completed_line_breaks"], team["matches"])
    team["completed_defensive_line_breaks_per_match"] = safe_divide(team["completed_defensive_line_breaks"], team["matches"])

    team["line_break_completion_rate"] = safe_divide(team["completed_line_breaks"], team["attempted_line_breaks"]) * 100
    team["defensive_line_break_completion_rate"] = safe_divide(
        team["completed_defensive_line_breaks"],
        team["attempted_defensive_line_breaks"]
    ) * 100
    team["defensive_line_break_share"] = safe_divide(
        team["completed_defensive_line_breaks"],
        team["completed_line_breaks"]
    ) * 100

    team["wide_entry_share"] = safe_divide(team["wide_entries"], team["final_third_entries_total"]) * 100
    team["halfspace_entry_share"] = safe_divide(team["halfspace_entries"], team["final_third_entries_total"]) * 100
    team["central_entry_share"] = safe_divide(team["central_entries"], team["final_third_entries_total"]) * 100

    team["in_behind_offer_share"] = safe_divide(team["offers_in_behind"], team["offers_total"]) * 100
    team["between_lines_offer_share"] = safe_divide(team["offers_in_between"], team["offers_total"]) * 100
    team["in_front_offer_share"] = safe_divide(team["offers_in_front"], team["offers_total"]) * 100
    team["in_behind_reception_rate"] = safe_divide(team["receptions_behind_line"], team["offers_in_behind"]) * 100
    team["between_lines_reception_rate"] = safe_divide(team["receptions_between_lines"], team["offers_in_between"]) * 100
    team["pressing_to_turnover_rate"] = safe_divide(team["forced_turnovers"], team["pressing_applied"]) * 100

    return team


metrics_overall = calculate_team_metrics(df, "Overall")
metrics_group_stage = calculate_team_metrics(df[df["competition_phase"] == "Group Stage"], "Group Stage")
metrics_knockout_stage = calculate_team_metrics(df[df["competition_phase"] == "Knockout Stage"], "Knockout Stage")

team_metrics_by_phase = pd.concat(
    [metrics_overall, metrics_group_stage, metrics_knockout_stage],
    ignore_index=True,
)

team_metrics_path = OUTPUT_DATA_DIR / "champion_profile_team_metrics_by_phase.csv"
team_metrics_by_phase.to_csv(team_metrics_path, index=False, encoding="utf-8-sig")

print(f"[Check] Saved team metrics: {team_metrics_path}")
display(
    team_metrics_by_phase
    .groupby("competition_phase")
    .agg(
        teams=("team_name", "nunique"),
        avg_matches=("matches", "mean"),
        min_matches=("matches", "min"),
        max_matches=("matches", "max"),
    )
    .round(2)
    .reset_index()
)




## Champion Ranking Table

The ranking table answers a simple report question:

> Where did Spain rank across key tactical indicators?

A rank of `1` means the highest value among teams in the selected phase. For style-share metrics, rank indicates tendency, not superiority.


In [ ]:
"""
Step 3: Build Spain ranking profile
===================================
Ranks are calculated within each competition phase.
"""

performance_metrics = {
    "field_tilt_proxy": "Field Tilt Proxy",
    "shot_creation_efficiency": "Shot Creation Efficiency",
    "goals_per_attempt": "Goals per Attempt",
    "final_third_entries_per_match": "Final Third Entries per Match",
    "attempts_at_goal_per_match": "Attempts at Goal per Match",
    "line_break_completion_rate": "Line Break Completion Rate",
    "defensive_line_break_completion_rate": "Defensive Line Break Completion Rate",
    "completed_line_breaks_per_match": "Completed Line Breaks per Match",
    "completed_defensive_line_breaks_per_match": "Completed Defensive Line Breaks per Match",
    "defensive_line_break_share": "Defensive Line Break Share",
    "pressing_to_turnover_rate": "Pressing-to-Turnover Rate",
}

style_metrics = {
    "wide_entry_share": "Wide Entry Share",
    "halfspace_entry_share": "Half-space Entry Share",
    "central_entry_share": "Central Entry Share",
    "in_behind_offer_share": "In-behind Offer Share",
    "between_lines_offer_share": "Between-lines Offer Share",
    "in_front_offer_share": "In-front Offer Share",
    "in_behind_reception_rate": "In-behind Reception Rate",
    "between_lines_reception_rate": "Between-lines Reception Rate",
}

metric_groups = {**{k: "Performance / Efficiency" for k in performance_metrics}, **{k: "Style Profile" for k in style_metrics}}
metric_labels = {**performance_metrics, **style_metrics}

ranking_rows = []

for phase, phase_df in team_metrics_by_phase.groupby("competition_phase"):
    phase_df = phase_df.copy()
    n_teams = phase_df["team_name"].nunique()

    if CHAMPION_TEAM not in set(phase_df["team_name"]):
        continue

    champion_row = phase_df.loc[phase_df["team_name"] == CHAMPION_TEAM].iloc[0]

    for metric, label in metric_labels.items():
        metric_rank = phase_df[metric].rank(ascending=False, method="min")
        champion_rank = int(metric_rank.loc[phase_df["team_name"] == CHAMPION_TEAM].iloc[0])
        champion_value = champion_row[metric]
        phase_median = phase_df[metric].median()
        phase_mean = phase_df[metric].mean()
        percentile = (n_teams - champion_rank + 1) / n_teams * 100

        ranking_rows.append({
            "competition_phase": phase,
            "metric_group": metric_groups[metric],
            "metric": metric,
            "metric_label": label,
            "champion": CHAMPION_TEAM,
            "champion_value": champion_value,
            "rank": champion_rank,
            "teams_in_phase": n_teams,
            "percentile_rank": percentile,
            "phase_median": phase_median,
            "phase_mean": phase_mean,
            "difference_vs_median": champion_value - phase_median,
        })

champion_rankings = pd.DataFrame(ranking_rows)
champion_rankings = champion_rankings.sort_values(
    ["competition_phase", "metric_group", "rank", "metric_label"]
)

rankings_path = OUTPUT_DATA_DIR / "champion_profile_spain_metric_rankings.csv"
champion_rankings.to_csv(rankings_path, index=False, encoding="utf-8-sig")

print(f"[Check] Saved champion rankings: {rankings_path}")

display(
    champion_rankings
    .assign(
        champion_value=lambda x: x["champion_value"].round(2),
        phase_median=lambda x: x["phase_median"].round(2),
        difference_vs_median=lambda x: x["difference_vs_median"].round(2),
        percentile_rank=lambda x: x["percentile_rank"].round(1),
    )
)




In [ ]:
"""
Step 4: Champion profile summary for the report
===============================================
This table is designed to be read directly when writing the report.
"""

phase_to_show = "Overall"
# phase_to_show = "Group Stage"
# phase_to_show = "Knockout Stage"

summary_cols = [
    "metric_group",
    "metric_label",
    "champion_value",
    "rank",
    "teams_in_phase",
    "percentile_rank",
    "phase_median",
    "difference_vs_median",
]

champion_profile_summary = (
    champion_rankings[champion_rankings["competition_phase"] == phase_to_show]
    [summary_cols]
    .copy()
    .sort_values(["metric_group", "rank"])
)

champion_profile_summary[["champion_value", "percentile_rank", "phase_median", "difference_vs_median"]] = (
    champion_profile_summary[["champion_value", "percentile_rank", "phase_median", "difference_vs_median"]]
    .round(2)
)

print(f"[Champion Profile: {phase_to_show}]")
display(champion_profile_summary)

summary_path = OUTPUT_DATA_DIR / f"champion_profile_spain_summary_{phase_to_show.lower().replace(' ', '_')}.csv"
champion_profile_summary.to_csv(summary_path, index=False, encoding="utf-8-sig")
print(f"Saved summary: {summary_path}")




## Visualization 1: Spain's Performance Ranking

This chart focuses on performance and efficiency indicators only. Higher percentile means Spain ranked closer to the top of the selected phase.


In [ ]:
"""
Step 5: Plot Spain's performance percentile rankings
====================================================
This version uses a clearer color system:
- Dark green: elite ranking, top quartile
- Teal: above median
- Amber: below median
- Red: bottom quartile
"""

plot_phase = "Overall"
# plot_phase = "Group Stage"
# plot_phase = "Knockout Stage"

plot_df = (
    champion_rankings[
        (champion_rankings["competition_phase"] == plot_phase)
        & (champion_rankings["metric_group"] == "Performance / Efficiency")
    ]
    .copy()
    .sort_values("percentile_rank", ascending=True)
)

if plot_df.empty:
    raise ValueError(f"No performance metrics found for phase: {plot_phase}")


def ranking_color(percentile):
    if percentile >= 75:
        return "#047857"  # elite / top quartile
    if percentile >= 50:
        return "#0f766e"  # above median
    if percentile >= 25:
        return "#f59e0b"  # below median
    return "#dc2626"      # bottom quartile

plot_df["bar_color"] = plot_df["percentile_rank"].apply(ranking_color)

fig, ax = plt.subplots(figsize=(11.8, 7.6), dpi=150)
fig.patch.set_facecolor("#fbfaf7")
ax.set_facecolor("#fbfaf7")

bars = ax.barh(
    plot_df["metric_label"],
    plot_df["percentile_rank"],
    color=plot_df["bar_color"],
    alpha=0.95,
    height=0.62,
)

# Quartile background guides
ax.axvspan(0, 25, color="#fee2e2", alpha=0.30, zorder=0)
ax.axvspan(25, 50, color="#fef3c7", alpha=0.30, zorder=0)
ax.axvspan(50, 75, color="#ccfbf1", alpha=0.26, zorder=0)
ax.axvspan(75, 100, color="#dcfce7", alpha=0.30, zorder=0)

ax.axvline(50, linestyle="--", color="#4b5563", linewidth=1.1, alpha=0.80, zorder=2)
ax.text(
    50,
    len(plot_df) - 0.25,
    "Median",
    ha="center",
    va="bottom",
    fontsize=8.5,
    color="#4b5563",
)

for y, (_, row) in enumerate(plot_df.iterrows()):
    label = f"#{int(row['rank'])} / {int(row['teams_in_phase'])}"
    value = row["percentile_rank"]

    # Put labels inside long bars and outside shorter bars.
    if value >= 72:
        x_pos = value - 2.0
        ha = "right"
        label_color = "white"
    else:
        x_pos = value + 1.4
        ha = "left"
        label_color = "#111827"

    ax.text(
        x_pos,
        y,
        label,
        va="center",
        ha=ha,
        fontsize=9,
        color=label_color,
        fontweight="medium",
    )

ax.set_xlim(0, 108)
ax.set_xlabel("Percentile Rank within Phase (higher = closer to #1)", fontsize=11, color="#111827")
ax.set_title(
    f"Spain Champion Profile: Performance Rankings ({plot_phase})",
    fontsize=15.5,
    pad=16,
    color="#111827",
)

ax.tick_params(axis="y", labelsize=10.5, colors="#111827")
ax.tick_params(axis="x", labelsize=9.5, colors="#374151")
ax.grid(axis="x", alpha=0.18, color="#9ca3af", linewidth=0.8)
ax.set_axisbelow(True)

for spine in ax.spines.values():
    spine.set_visible(False)

# Compact legend-like note
legend_text = "Top quartile   Above median   Below median   Bottom quartile"
fig.text(0.08, 0.086, legend_text, fontsize=8.5, color="#4b5563")

footnote = (
    "Rank #1 means the highest value in the selected phase. Rankings are descriptive, not causal.\n"
    "Source: FIFA Match Centre | Full Official Stats only.\nNote: descriptive rankings; match counts 3-8; Belgium/Egypt each miss one full-stat match."
)
fig.text(0.08, 0.030, footnote, fontsize=7.8, color="#6b7280", linespacing=1.18)

plt.tight_layout(rect=[0, 0.105, 1, 1])

output_path = OUTPUT_FIG_DIR / f"07_champion_profile_spain_performance_rankings_{plot_phase.lower().replace(' ', '_')}.png"
fig.savefig(output_path, dpi=220, bbox_inches="tight", facecolor=fig.get_facecolor())
plt.show()

print(f"Saved figure: {output_path}")






## Visualization 2: Spain's Style Profile

This chart shows how Spain's tactical tendencies ranked. These are not automatically “better” or “worse”; they describe how Spain played relative to the tournament field.


In [ ]:
"""
Step 6: Plot Spain's style profile rankings
===========================================
This version colors each bar by tactical category:
- Entry route: how Spain entered the final third
- Off-ball offer: where players offered to receive
- Reception access: whether offers became receptions
"""

plot_phase = "Overall"
# plot_phase = "Group Stage"
# plot_phase = "Knockout Stage"

style_plot_df = (
    champion_rankings[
        (champion_rankings["competition_phase"] == plot_phase)
        & (champion_rankings["metric_group"] == "Style Profile")
    ]
    .copy()
    .sort_values("percentile_rank", ascending=True)
)

if style_plot_df.empty:
    raise ValueError(f"No style metrics found for phase: {plot_phase}")


def style_category(metric):
    if metric in ["wide_entry_share", "halfspace_entry_share", "central_entry_share"]:
        return "Entry route"
    if metric in ["in_behind_offer_share", "between_lines_offer_share", "in_front_offer_share"]:
        return "Off-ball offer"
    if metric in ["in_behind_reception_rate", "between_lines_reception_rate"]:
        return "Reception access"
    return "Other"

category_colors = {
    "Entry route": "#7c3aed",       # violet
    "Off-ball offer": "#0891b2",    # cyan
    "Reception access": "#ea580c",  # orange
    "Other": "#6b7280",
}

style_plot_df["style_category"] = style_plot_df["metric"].apply(style_category)
style_plot_df["bar_color"] = style_plot_df["style_category"].map(category_colors)

fig, ax = plt.subplots(figsize=(11.8, 6.8), dpi=150)
fig.patch.set_facecolor("#fbfaf7")
ax.set_facecolor("#fbfaf7")

# Light background bands help read percentile position without implying good/bad.
ax.axvspan(0, 50, color="#f3f4f6", alpha=0.55, zorder=0)
ax.axvspan(50, 100, color="#ecfeff", alpha=0.38, zorder=0)

ax.barh(
    style_plot_df["metric_label"],
    style_plot_df["percentile_rank"],
    color=style_plot_df["bar_color"],
    alpha=0.92,
    height=0.62,
)

for y, (_, row) in enumerate(style_plot_df.iterrows()):
    label = f"#{int(row['rank'])} / {int(row['teams_in_phase'])}"
    value = row["percentile_rank"]

    if value >= 72:
        x_pos = value - 2.0
        ha = "right"
        label_color = "white"
    else:
        x_pos = value + 1.4
        ha = "left"
        label_color = "#111827"

    ax.text(
        x_pos,
        y,
        label,
        va="center",
        ha=ha,
        fontsize=9,
        color=label_color,
        fontweight="medium",
    )

ax.axvline(50, linestyle="--", color="#4b5563", linewidth=1.1, alpha=0.80)
ax.text(
    50,
    len(style_plot_df) - 0.25,
    "Median tendency",
    ha="center",
    va="bottom",
    fontsize=8.5,
    color="#4b5563",
)

ax.set_xlim(0, 108)
ax.set_xlabel("Percentile Rank within Phase (higher = stronger tendency)", fontsize=11, color="#111827")
ax.set_title(
    f"Spain Champion Profile: Style Tendencies ({plot_phase})",
    fontsize=15.5,
    pad=16,
    color="#111827",
)

ax.tick_params(axis="y", labelsize=10.5, colors="#111827")
ax.tick_params(axis="x", labelsize=9.5, colors="#374151")
ax.grid(axis="x", alpha=0.18, color="#9ca3af", linewidth=0.8)
ax.set_axisbelow(True)

for spine in ax.spines.values():
    spine.set_visible(False)

# Category legend
legend_handles = [
    plt.Line2D([0], [0], marker="s", color="none", markerfacecolor=color, markersize=8, label=label)
    for label, color in category_colors.items()
    if label in set(style_plot_df["style_category"])
]

ax.legend(
    handles=legend_handles,
    loc="lower right",
    frameon=False,
    fontsize=8.5,
    title="Metric category",
    title_fontsize=8.5,
)

footnote = (
    "Style metrics show tendency, not quality.\n"
    "Source: FIFA Match Centre | Full Official Stats only.\nNote: descriptive rankings; match counts 3-8; Belgium/Egypt each miss one full-stat match."
)
fig.text(0.08, 0.030, footnote, fontsize=7.8, color="#6b7280", linespacing=1.18)

plt.tight_layout(rect=[0, 0.105, 1, 1])

output_path = OUTPUT_FIG_DIR / f"08_champion_profile_spain_style_rankings_{plot_phase.lower().replace(' ', '_')}.png"
fig.savefig(output_path, dpi=220, bbox_inches="tight", facecolor=fig.get_facecolor())
plt.show()

print(f"Saved figure: {output_path}")






## Report Writing Notes

Use the final interpretation in three parts:

1. **What Spain ranked highly in**  
   These are the areas where the champion clearly stood out relative to the field.

2. **What Spain was average or lower in**  
   These are not weaknesses automatically. They may show that Spain did not need to dominate every tactical indicator to win.

3. **What the data cannot prove**  
   FIFA Official Stats do not include xG in this dataset, so this analysis cannot directly measure chance quality. Rankings are descriptive and should be combined with match context.


In [ ]:
"""
Step 7: Generate concise written takeaways
==========================================
This cell creates draft text that can be edited for a LinkedIn report or slide notes.
"""

report_phase = "Overall"
report_df = champion_rankings[champion_rankings["competition_phase"] == report_phase].copy()

performance_report = report_df[report_df["metric_group"] == "Performance / Efficiency"].sort_values("rank")
style_report = report_df[report_df["metric_group"] == "Style Profile"].sort_values("rank")

top_performance = performance_report.head(4)
lower_performance = performance_report.tail(3).sort_values("rank")
top_style = style_report.head(4)

print(f"Champion Profile Draft Notes: {CHAMPION_TEAM} ({report_phase})")
print("=" * 72)

print("\nTop performance / efficiency rankings:")
for _, row in top_performance.iterrows():
    print(f"- {row['metric_label']}: #{int(row['rank'])} of {int(row['teams_in_phase'])} ({row['champion_value']:.2f})")

print("\nLower or more average performance rankings:")
for _, row in lower_performance.iterrows():
    print(f"- {row['metric_label']}: #{int(row['rank'])} of {int(row['teams_in_phase'])} ({row['champion_value']:.2f})")

print("\nStrongest style tendencies:")
for _, row in top_style.iterrows():
    print(f"- {row['metric_label']}: #{int(row['rank'])} of {int(row['teams_in_phase'])} ({row['champion_value']:.2f})")

print("\nSuggested interpretation:")
print(
    "Spain's champion profile should be read as a combination of strengths and tactical tendencies. "
    "High rankings show where Spain stood out relative to the tournament field, while lower rankings show that the champion did not need to lead every metric. "
    "Because this dataset does not include xG, the analysis describes territory, progression, and shot volume rather than chance quality."
)


